<!-- SPDX-License-Identifier: CC-BY-4.0 -->
<!-- Copyright Contributors to the ODPi Egeria project. -->

![Egeria Logo](https://raw.githubusercontent.com/odpi/egeria/main/assets/img/ODPi_Egeria_Logo_color.png)

### Egeria Workbook

# Building the Egeria operational data store

Egeria publishes extracts of the open metadata ecosystem as [digital products](https://egeria-project.org/concepts/digital-product/): each product is a table of values, such as the list of departments, the valid metadata values or the latest survey reports.  They are catalogued in the *Open Metadata Digital Product Catalog* (see the [open-metadata-product-catalog](../open-metadata-product-catalog/README.md) workbooks).

This workbook builds an operational data store for these products: a PostgreSQL database called `egeria_ods` that holds one table for each product, refreshed once a day.  With it, the content of the open metadata ecosystem can be queried with SQL, joined, and used by reporting tools.

The products are organized into product families.  Subscribing to a family delivers every product in it - including the products of the families nested inside it, and products that are added later.  The families sit in a catalog folder called **Digital Products**.  The folder is not itself a product, so it cannot be subscribed to; instead the workbook subscribes to each of the families directly beneath it, and gives each family its own schema in the database.

The workbook does three things.  Each is skipped if it has already been done, so the workbook can be run again at any time - for example, to pick up a family that has been added to the folder.  It is written for the *freshstart* environment; the quickstart environment builds the same database from a Dr.Egeria file in `coco-workbooks/1. coco-data-hub`.

1. It checks that the `egeria_ods` database exists and makes sure it is catalogued in Egeria.
2. It makes sure each family has a destination - a schema in `egeria_ods` - described in Egeria.
3. It takes out a daily subscription to each family, with the family's schema as the destination.

Egeria's provisioning services then create the schemas and their tables, and fill them.

----

In [ ]:
# Initialize pyegeria

import os
import re
import time
from datetime import datetime

import pandas
import psycopg2
from IPython.display import display, Markdown

from pyegeria import (
    AutomatedCuration, ClassificationExplorer, CollectionManager, MyProfile, ProductManager,
    settings,
)

env = settings.Environment
view_server = env.egeria_view_server
url = env.egeria_view_server_url
user_id = os.environ.get("EGERIA_USER", "erinoverview")
user_pwd = os.environ.get("EGERIA_USER_PASSWORD", "secret")

print("Accessing view server " + view_server + " on platform " + url + " for user " + user_id)

# The classification explorer client finds elements of any type
classification_explorer = ClassificationExplorer(view_server, url, user_id, user_pwd)

# The automated curation client creates elements from templates and runs governance action processes
automated_curation = AutomatedCuration(view_server, url, user_id, user_pwd)

# The collection manager client retrieves the members of the Digital Products folder
collection_manager = CollectionManager(view_server, url, user_id, user_pwd)

# The product manager client retrieves digital products and their subscription types
product_manager = ProductManager(view_server, url, user_id, user_pwd)

# The my profile client retrieves the profile of the user running this workbook
my_profile = MyProfile(view_server, url, user_id, user_pwd)

NOT_FOUND = "No elements found"


def refresh_tokens():
    """Get a new security token and share it between the clients."""
    token = classification_explorer.create_egeria_bearer_token()
    for client in (automated_curation, collection_manager, product_manager, my_profile):
        client.set_bearer_token(token)


def find_guid(qualified_name):
    """Return the unique identifier (guid) of the element with this qualified name, or None if there is none."""
    guid = classification_explorer.get_element_guid_by_unique_name(qualified_name)
    return guid if isinstance(guid, str) and guid != NOT_FOUND else None


def wait_for_element(qualified_name, timeout_seconds=300):
    """Wait for an element that a background process is creating, returning its guid, or None if it does not appear."""
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        guid = find_guid(qualified_name)
        if guid:
            return guid
        time.sleep(5)
    return None


refresh_tokens()

----

## The values used in this workbook

The cell below names the database, the folder that holds the product families and the type of subscription.  The defaults are what the workbook is designed for; each can be changed through an environment variable for an experiment.

The PostgreSQL server is described twice because it is reached in two ways: Egeria connects to it through `host.docker.internal`, and this workbook connects to it over the container network.

----

In [ ]:
# The database that receives the products
ods_database_name = os.environ.get("EGERIA_ODS_DATABASE", "egeria_ods")

# The folder whose product families are subscribed to, and the type of subscription.  The folder was catalogued
# as a product family by earlier releases of Egeria, so both of its qualified names are tried.
product_folder_qualified_names = [
    "DigitalProductCatalog::Jacquard::Open Metadata Digital Products",
    "DigitalProductFamily::Jacquard::Open Metadata Digital Products",
]
if os.environ.get("EGERIA_ODS_PRODUCT_FOLDER"):
    product_folder_qualified_names = [os.environ["EGERIA_ODS_PRODUCT_FOLDER"]]

subscription_type_identifier = os.environ.get("EGERIA_ODS_SUBSCRIPTION_TYPE", "DAILY-REFRESH-SUBSCRIPTION")

# How Egeria reaches the PostgreSQL server, and the secrets it uses
postgres_host_identifier = "host.docker.internal"
postgres_port_number = "5442"
secrets_store_path_name = "secrets/integration.omsecrets"
cataloguing_secrets_collection_name = "PostgreSQL Server Secret"
provisioning_secrets_collection_name = "PostgreSQL Provisioning Secret"

# How this workbook reaches the PostgreSQL server, to check the database and to query the products
workbook_postgres_host = os.environ.get("EGERIA_ODS_POSTGRES_HOST", "egeria-shared-postgres")
workbook_postgres_user = os.environ.get("EGERIA_ODS_POSTGRES_USER", "egeria_user")
workbook_postgres_password = os.environ.get("EGERIA_ODS_POSTGRES_PASSWORD", "user4egeria")

# The name that the PostgreSQL server is catalogued under
postgres_server_name = os.environ.get("EGERIA_ODS_POSTGRES_SERVER_NAME", "Egeria Shared PostgreSQL Server")

database_qualified_name = "PostgreSQL Relational Database::" + postgres_server_name + "::" + ods_database_name


def schema_name_for(family_name):
    """Return the name of the schema that receives a product family: its name in lower case, with underscores."""
    return re.sub(r"[^a-z0-9]+", "_", family_name.lower()).strip("_")


def destination_qualified_name_for(schema_name):
    """Return the qualified name of the destination data set that describes a schema of the database."""
    return ("PostgreSQL Tabular Data Set Collection::" + postgres_server_name + "::" +
            ods_database_name + "." + schema_name)


print("PostgreSQL server: " + postgres_server_name)
print("Database:          " + database_qualified_name)
print("Subscription type: " + subscription_type_identifier)

----

## Checking the database

The `egeria_ods` database is created with the shared PostgreSQL server that the freshstart environment uses.  It starts empty: Egeria's provisioning services create the schemas and the tables when they first deliver the products.

The cell below connects to the database and lists the tables it has so far.  If the database is missing, restart the environment - the start-up scripts add it to a PostgreSQL server that was created before the database was introduced.

----

In [ ]:
def query_ods(sql, parameters=None):
    """Run a query against the operational data store and return the result as a data frame."""
    connection = psycopg2.connect(host=workbook_postgres_host, port=postgres_port_number, dbname=ods_database_name,
                                  user=workbook_postgres_user, password=workbook_postgres_password)
    try:
        with connection.cursor() as cursor:
            cursor.execute(sql, parameters)
            return pandas.DataFrame(cursor.fetchall(), columns=[column.name for column in cursor.description])
    finally:
        connection.close()


def list_ods_tables():
    """Return the product tables in the database, with the schema each is in and the number of rows it has."""
    tables = query_ods("select table_schema, table_name from information_schema.tables " +
                       "where table_schema not like 'pg\\_%' and table_schema not in ('information_schema', 'public') " +
                       "order by table_schema, table_name")
    row_counts = []
    for schema_name, table_name in zip(tables["table_schema"], tables["table_name"]):
        counted = query_ods('select count(*) as row_count from "' + schema_name + '"."' + table_name + '"')
        row_counts.append(int(counted["row_count"][0]))
    tables["row_count"] = row_counts
    return tables


try:
    ods_tables = list_ods_tables()
    print("Connected to " + ods_database_name + ": it has " + str(len(ods_tables)) + " product table(s).")
except psycopg2.OperationalError as error:
    raise RuntimeError("The " + ods_database_name + " database is not available on " + workbook_postgres_host + ":" +
                       postgres_port_number + ".  Restart the environment so that its start-up scripts create it, " +
                       "then run this workbook again.") from error

----

## Cataloguing the database

Egeria needs a description of the database before it can deliver data to it.  The governance action process called `PostgreSQLDatabase::CreateAsCatalogTargetGovernanceActionProcess` creates the description and attaches it to the integration connector that catalogues databases, so that the schemas, tables and columns that appear in `egeria_ods` are catalogued too, and kept up to date.

The process runs in the background.  The cell below starts it - unless the database is already catalogued - and waits for the description of the database to appear.

----

In [ ]:
refresh_tokens()

database_guid = find_guid(database_qualified_name)

if database_guid:
    print("The database is already catalogued: " + database_guid)
else:
    create_and_catalog_process_name = "PostgreSQLDatabase::CreateAsCatalogTargetGovernanceActionProcess"

    if not find_guid(create_and_catalog_process_name):
        raise RuntimeError("The " + create_and_catalog_process_name + " process is not defined.  Load the " +
                           "PostgresContentPack (see the loading-metadata workbooks) and run this workbook again.")

    request_parameters = {
        "serverName" : postgres_server_name,
        "hostIdentifier" : postgres_host_identifier,
        "portNumber" : postgres_port_number,
        "secretsStorePathName" : secrets_store_path_name,
        "secretsCollectionName" : cataloguing_secrets_collection_name,
        "versionIdentifier" : "1.0",
        "databaseName" : ods_database_name,
        "databaseDescription" : "Egeria's operational data store: one table for each open metadata digital product, " +
                                "refreshed by subscriptions to the product families of the Open Metadata Digital " +
                                "Product Catalog."
    }

    print("Starting the " + create_and_catalog_process_name + " process")
    automated_curation.initiate_gov_action_process(action_type_qualified_name=create_and_catalog_process_name,
                                                   request_source_guids=None,
                                                   action_targets=None,
                                                   start_time=datetime.now(),
                                                   request_parameters=request_parameters,
                                                   orig_service_name=None,
                                                   orig_engine_name=None)

    database_guid = wait_for_element(database_qualified_name)

    if database_guid:
        print("The database is now catalogued: " + database_guid)
    else:
        raise RuntimeError("The database has not been catalogued after five minutes.  Check the engine actions " +
                           "for the " + create_and_catalog_process_name + " process, then run this cell again.")

----

## Finding the product families

The cell below retrieves the *Digital Products* folder and lists the product families directly beneath it.

Each product family lists the types of subscription it offers.  A subscription type is a governance action process: running it, with the subscriber and the destination as its action targets, creates the subscription.  The cell looks through each family's subscription types for the daily refresh.

----

In [ ]:
refresh_tokens()

product_folder_guid = None
for product_folder_qualified_name in product_folder_qualified_names:
    product_folder_guid = find_guid(product_folder_qualified_name)
    if product_folder_guid:
        break

# Each entry describes one product family: its name, its guid, the schema that receives it and the process
# that creates a daily refresh subscription to it (None if the family does not offer one).
product_families = []

if not product_folder_guid:
    print("The Digital Products folder is not in the product catalog yet.")
    print("The catalog is built by the Jacquard digital product harvester, which runs in the nanny daemon.")
else:
    product_folder = collection_manager.get_collection_by_guid(guid=product_folder_guid, output_format="JSON")

    for member in product_folder.get("collectionMembers") or []:
        related_element = member.get("relatedElement", {})
        if related_element.get("elementHeader", {}).get("type", {}).get("typeName") != "DigitalProductFamily":
            continue

        family_guid = related_element["elementHeader"]["guid"]
        family_name = related_element.get("properties", {}).get("displayName", family_guid)

        # A product's subscription types are in its resource list, as the processes that create a subscription
        family = product_manager.get_digital_product_by_guid(family_guid, output_format="JSON")
        subscription_process_qualified_name = None
        for resource in family.get("resourceList") or []:
            resource_element = resource.get("relatedElement", {})
            resource_name = resource_element.get("properties", {}).get("qualifiedName", "")
            if (resource_element.get("elementHeader", {}).get("type", {}).get("typeName") == "SubscribingActionProcess"
                    and resource_name.endswith("::" + subscription_type_identifier)):
                subscription_process_qualified_name = resource_name

        product_families.append({"name" : family_name,
                                 "guid" : family_guid,
                                 "schema" : schema_name_for(family_name),
                                 "subscription_process" : subscription_process_qualified_name})

    product_families.sort(key=lambda product_family: product_family["name"])

    print("The Digital Products folder has " + str(len(product_families)) + " product families:")
    for product_family in product_families:
        if product_family["subscription_process"]:
            print("  " + product_family["name"] + " -> schema " + product_family["schema"])
        else:
            print("  " + product_family["name"] + " - does not offer the " + subscription_type_identifier +
                  " subscription type, so it is skipped")

----

## Describing the destinations and subscribing

A subscription delivers a product to a *destination data set*.  For a single product the destination is one table.  For a product family it is a collection of tables: a database schema that receives one table for each product in the family.

For each family, the cell below:

* creates the description of the family's schema from the *PostgreSQL Tabular Data Set Collection* template, unless it already exists.  The description includes the connection that the provisioning services use, with the `provisioner` database user from the *PostgreSQL Provisioning Secret*.  The schema itself is not created yet.
* subscribes the destination to the family, unless the destination already has a subscription.  The subscription is made on behalf of the person running this workbook: their profile is the *subscription requester*, and the destination data set is the *subscriber*.

----

In [ ]:
refresh_tokens()


def get_subscriptions(data_set_guid):
    """Return the guids of the digital subscriptions that deliver to a destination data set."""
    subscribers = classification_explorer.get_related_elements(data_set_guid, "DigitalSubscriber")
    if not isinstance(subscribers, list):
        return []
    return [subscriber.get("relatedElement", subscriber)["elementHeader"]["guid"] for subscriber in subscribers]


def get_destination(product_family):
    """Return the guid of the destination data set for a product family, creating its description if needed."""
    destination_qualified_name = destination_qualified_name_for(product_family["schema"])
    destination_guid = find_guid(destination_qualified_name)
    if destination_guid:
        return destination_guid

    body = {
        "class" : "TemplateRequestBody",
        "typeName" : "TabularDataSetCollection",
        "templateGUID" : template_guid,
        "isOwnAnchor" : True,
        "placeholderPropertyValues" : {
            "hostIdentifier" : postgres_host_identifier,
            "portNumber" : postgres_port_number,
            "serverName" : postgres_server_name,
            "secretsStorePathName" : secrets_store_path_name,
            "secretsCollectionName" : provisioning_secrets_collection_name,
            "versionIdentifier" : "V1.0",
            "databaseName" : ods_database_name,
            "schemaName" : product_family["schema"],
            "schemaDescription" : "The products of the " + product_family["name"] + " product family, one table " +
                                  "for each product, delivered by a daily subscription to the family."
        }
    }
    destination_guid = automated_curation.create_elem_from_template(body)
    print("  described the destination " + destination_qualified_name)
    return destination_guid


template_guid = find_guid("PostgreSQL Tabular Data Set Collection::~{serverName}~::~{databaseName}~.~{schemaName}~")
if not template_guid:
    raise RuntimeError("The PostgreSQL Tabular Data Set Collection template is not defined.  Load the " +
                       "PostgresContentPack (see the loading-metadata workbooks) and run this workbook again.")

requester = my_profile.get_my_profile()
requester_guid = requester["elementHeader"]["guid"]
print("Subscribing on behalf of " + requester["properties"].get("qualifiedName", requester_guid))

# Start the subscription process for each family that needs one
waiting_for_subscription = {}

for product_family in product_families:
    if not product_family["subscription_process"]:
        continue

    print(product_family["name"])
    destination_guid = get_destination(product_family)

    if get_subscriptions(destination_guid):
        print("  already subscribed")
        continue

    action_targets = [
        {"class" : "NewActionTarget", "actionTargetName" : "digitalSubscriptionRequester", "actionTargetGUID" : requester_guid},
        {"class" : "NewActionTarget", "actionTargetName" : "destinationDataSet", "actionTargetGUID" : destination_guid}
    ]
    automated_curation.initiate_gov_action_process(action_type_qualified_name=product_family["subscription_process"],
                                                   request_source_guids=None,
                                                   action_targets=action_targets,
                                                   start_time=datetime.now(),
                                                   request_parameters=None,
                                                   orig_service_name=None,
                                                   orig_engine_name=None)
    print("  started " + product_family["subscription_process"])
    waiting_for_subscription[product_family["name"]] = destination_guid

# The processes run in the background: wait for the subscriptions they create
deadline = time.time() + 300
while waiting_for_subscription and time.time() < deadline:
    time.sleep(5)
    for family_name, destination_guid in list(waiting_for_subscription.items()):
        if get_subscriptions(destination_guid):
            print("Subscribed to " + family_name)
            del waiting_for_subscription[family_name]

for family_name in waiting_for_subscription:
    print("The subscription to " + family_name + " has not appeared after five minutes.  Check the engine " +
          "actions for its process.")

----

## Reviewing the operational data store

A daily refresh subscription delivers the family's products once a day, starting from the time of the subscription.  The cell below lists the tables that have been delivered so far, with the number of rows in each.  Run it again later to see the database fill.

A product whose data cannot be delivered is skipped, so that the rest of its family still arrives.

----

In [ ]:
ods_tables = list_ods_tables()

if len(ods_tables) == 0:
    print("No products have been delivered to " + ods_database_name + " yet.")
else:
    print(str(len(ods_tables)) + " product table(s) in " + ods_database_name + ":")
    display(ods_tables)

----

Each table can be queried with SQL.  The cell below shows the first rows of the first table; change the query to explore the others.

----

In [ ]:
if len(ods_tables) > 0:
    first_schema_name = ods_tables["table_schema"][0]
    first_table_name = ods_tables["table_name"][0]
    print(first_schema_name + "." + first_table_name)
    display(query_ods('select * from "' + first_schema_name + '"."' + first_table_name + '" limit 10'))

----

## Where next?

* [Reviewing my subscriptions](../open-metadata-product-catalog/reviewing-my-subscriptions.ipynb) shows how to review and cancel a subscription.
* [Searching the Open Metadata Digital Product Catalog](../open-metadata-product-catalog/searching-product-catalog.ipynb) describes the products that are delivered to the tables.

----
License: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/),
Copyright Contributors to the ODPi Egeria project.